In [1]:
import sys
sys.path.append('D:/cxr-triage')

mods_to_remove = [key for key in sys.modules if 'src' in key]
for mod in mods_to_remove:
    del sys.modules[mod]

import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve
from tqdm import tqdm
from src.data.dataset import ChestXrayDataset
from src.data.transforms import get_train_transforms, get_val_transforms
from src.models.densenet import DenseNetModel

LABELS = [
    'Atelectasis', 'Consolidation', 'Infiltration',
    'Pneumothorax', 'Edema', 'Emphysema', 'Fibrosis',
    'Effusion', 'Pneumonia', 'Pleural_Thickening',
    'Cardiomegaly', 'Nodule', 'Mass', 'Hernia'
]
CRITICAL = ['Pneumothorax', 'Edema', 'Pneumonia', 'Hernia']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMAGE_ROOT = "F:/X ray dataset/Second Version"

val_df = pd.read_csv('D:/cxr-triage/data/processed/val.csv')
test_df = pd.read_csv('D:/cxr-triage/data/processed/test.csv')


def load_model(checkpoint_path):
    model = DenseNetModel(num_classes=14, pretrained=False).to(DEVICE)
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"Loaded: epoch {checkpoint['epoch']+1}, train-time best AUC {checkpoint['best_auc']:.4f}")
    return model


def get_predictions(model, df, image_size=224, use_clahe=False, batch_size=8):
    transform = get_val_transforms(image_size=image_size, use_clahe=use_clahe)
    dataset = ChestXrayDataset(csv_path=None, image_root=IMAGE_ROOT, transform=transform)
    dataset.df = df
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    all_logits, all_targets = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Inference"):
            images = images.to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits = model(images)
            all_logits.append(logits.cpu().numpy())
            all_targets.append(targets.numpy())

    logits = np.concatenate(all_logits, axis=0).astype(np.float32)
    targets = np.concatenate(all_targets, axis=0).astype(np.float32)
    probs = 1 / (1 + np.exp(-logits))
    return probs, targets


def find_thresholds(val_probs, val_targets):
    thresholds = {}
    for i, label in enumerate(LABELS):
        precisions, recalls, thresh = precision_recall_curve(val_targets[:, i], val_probs[:, i])
        f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
        thresholds[label] = float(thresh[np.argmax(f1s)])
    return thresholds


def evaluate_full(checkpoint_path, image_size=224, use_clahe=False, batch_size=8):
    print(f"\n{'='*70}\nEvaluating: {checkpoint_path}\n{'='*70}")
    model = load_model(checkpoint_path)

    val_probs, val_targets = get_predictions(model, val_df, image_size, use_clahe, batch_size)
    thresholds = find_thresholds(val_probs, val_targets)

    test_probs, test_targets = get_predictions(model, test_df, image_size, use_clahe, batch_size)
    test_pred = np.zeros_like(test_probs, dtype=np.float32)
    for i, label in enumerate(LABELS):
        test_pred[:, i] = (test_probs[:, i] >= thresholds[label]).astype(np.float32)

    mean_auc = np.mean([roc_auc_score(test_targets[:, i], test_probs[:, i]) for i in range(14)])
    critical_auc = np.mean([roc_auc_score(test_targets[:, i], test_probs[:, i])
                             for i, l in enumerate(LABELS) if l in CRITICAL])
    micro_f1 = f1_score(test_targets, test_pred, average='micro')
    macro_f1 = f1_score(test_targets, test_pred, average='macro')

    print(f"\nMean Test AUC:     {mean_auc:.4f}")
    print(f"Critical Test AUC: {critical_auc:.4f}")
    print(f"Micro F1:          {micro_f1:.4f}")
    print(f"Macro F1:          {macro_f1:.4f}")

    per_class = {}
    for i, label in enumerate(LABELS):
        per_class[label] = {
            'AUC': roc_auc_score(test_targets[:, i], test_probs[:, i]),
            'F1': f1_score(test_targets[:, i], test_pred[:, i], zero_division=0),
            'Precision': precision_score(test_targets[:, i], test_pred[:, i], zero_division=0),
            'Recall': recall_score(test_targets[:, i], test_pred[:, i], zero_division=0),
            'Support': int(test_targets[:, i].sum())
        }

    return {
        'mean_auc': mean_auc, 'critical_auc': critical_auc,
        'micro_f1': micro_f1, 'macro_f1': macro_f1,
        'thresholds': thresholds, 'per_class': per_class
    }


print("Evaluation functions ready. Call evaluate_full(checkpoint_path, image_size, use_clahe) for each model.")

Evaluation functions ready. Call evaluate_full(checkpoint_path, image_size, use_clahe) for each model.


In [2]:
bce_results = evaluate_full(
    checkpoint_path='D:/cxr-triage/checkpoints/densenet_bce_fixed/best_model.pth',
    image_size=224,
    use_clahe=False
)


Evaluating: D:/cxr-triage/checkpoints/densenet_bce_fixed/best_model.pth
Loaded: epoch 16, train-time best AUC 0.7855


Inference: 100%|██████████| 3200/3200 [04:17<00:00, 12.43it/s]



Mean Test AUC:     0.7443
Critical Test AUC: 0.7730
Micro F1:          0.3215
Macro F1:          0.2479
